# Exploratory Data Analysis: Predicting School Performance in England (ANN)
**MRes First Progress Assessment — EDA Amendments**

This notebook implements the EDA stages required by the assessors:

1. **Data merging / homogenisation strategy** (Sections 3–5)
2. **Data statistical profiles with exemplary correlations** (Sections 6–7)
3. **Data quality statements** (Section 8)

Scope: state-funded **secondary** schools in **England**, school-level data.


## 1. Business / Research Understanding

Following the CRISP-DM style used in the ADS practicals.

In [ ]:
# DO NOT CHANGE THE VARIABLE NAME
business_understanding = {
    "stakeholder": (
        "Department for Education, Ofsted, local authorities and school "
        "leaders who allocate support and funding to state-funded secondary "
        "schools in England."
    ),
    "policy_problem": (
        "School performance varies with socio-economic context, but current "
        "monitoring (periodic inspection, published league tables) identifies "
        "struggling schools late. A predictive, interpretable model could "
        "flag at-risk schools earlier."
    ),
    "analytical_objective": (
        "Integrate DfE performance tables, Ofsted inspection outcomes and "
        "ONS/IMD socio-economic indicators at school level, and train an ANN "
        "to predict school performance (Ofsted overall effectiveness / "
        "Progress 8), explained with SHAP and LIME."
    ),
    "success_criteria": (
        "A homogenised school-level dataset covering >90% of eligible "
        "secondary schools; an ANN that outperforms linear and tree-based "
        "baselines on macro-F1 (classification) or RMSE (regression); "
        "explanations consistent with the education literature."
    ),
}
business_understanding


## 2. Data Acquisition

All datasets are open UK government data. Download the following files into `datasets/` **before running this notebook** (filenames below are the ones the loader expects — rename if your downloads differ):

| # | Dataset | Source / where to download | Expected filename |
|---|---------|---------------------------|-------------------|
| 1 | **GIAS** — Get Information About Schools (all establishments) | get-information-schools.service.gov.uk → Download → "All establishment data" | `datasets/gias_establishments.csv` |
| 2 | **DfE KS4 performance tables** (Progress 8 / Attainment 8, 2022-23 and 2023-24) | compare-school-performance.service.gov.uk → Download data → "Final KS4 data" per year | `datasets/ks4_2023.csv`, `datasets/ks4_2024.csv` |
| 3 | **Ofsted management information** — state-funded schools (latest monthly extract) | gov.uk → "State-funded school inspections and outcomes: management information" | `datasets/ofsted_mi.csv` |
| 4 | **IMD 2019** — File 7: all ranks, deciles and scores by LSOA | gov.uk → "English indices of deprivation 2019" | `datasets/imd2019.csv` |
| 5 | *(optional)* **Absence** — pupil absence in schools in England (school-level underlying data) | explore-education-statistics.service.gov.uk | `datasets/absence.csv` |

**Why GIAS matters for homogenisation:** GIAS provides, per school, the `URN`, phase, status, **postcode and the LSOA code** — so it acts as the *bridge* between DfE/Ofsted data (keyed on URN) and IMD data (keyed on LSOA), without needing the full ONS Postcode Directory.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
DATA = Path("datasets")

def load_csv(path, **kw):
    """Robust loader: keeps URN/LSOA codes as strings, tolerates encodings."""
    for enc in ("utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False,
                               dtype=str, **kw)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"Could not decode {path}")

gias   = load_csv(DATA / "gias_establishments.csv")
ks4_23 = load_csv(DATA / "ks4_2023.csv")
ks4_24 = load_csv(DATA / "ks4_2024.csv")
ofsted = load_csv(DATA / "ofsted_mi.csv")
imd    = load_csv(DATA / "imd2019.csv")

for name, df in [("GIAS", gias), ("KS4 2022/23", ks4_23),
                 ("KS4 2023/24", ks4_24), ("Ofsted MI", ofsted),
                 ("IMD 2019", imd)]:
    print(f"{name:<12} rows={len(df):>7,}  cols={df.shape[1]}")


### 2.1 Column harmonisation

Government files change column names between releases (e.g. `URN` vs `School URN`, `LSOA (code)` vs `LSOA code (2011)`). We normalise names once, so the rest of the notebook is release-independent — this is part of the **homogenisation strategy**.

In [ ]:
def norm_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                    .str.replace(r"[^a-z0-9]+", "_", regex=True)
                    .str.strip("_"))
    return df

gias, ks4_23, ks4_24, ofsted, imd = map(norm_cols, [gias, ks4_23, ks4_24, ofsted, imd])

def find_col(df, *candidates, contains=None):
    """Locate a column by exact candidates first, then substring."""
    for c in candidates:
        if c in df.columns:
            return c
    if contains:
        hits = [c for c in df.columns if contains in c]
        if hits:
            return hits[0]
    raise KeyError(f"None of {candidates} (or *{contains}*) found")

URN_GIAS   = find_col(gias, "urn")
LSOA_GIAS  = find_col(gias, "lsoa_code", contains="lsoa")
URN_KS4    = find_col(ks4_23, "urn")
URN_OFSTED = find_col(ofsted, "urn", contains="urn")
LSOA_IMD   = find_col(imd, "lsoa_code_2011", contains="lsoa_code")

print("Key columns resolved:",
      dict(gias=URN_GIAS, gias_lsoa=LSOA_GIAS, ks4=URN_KS4,
           ofsted=URN_OFSTED, imd=LSOA_IMD))


## 3. Defining the Analysis Population (Homogenisation step 1)

Research boundary: **open, state-funded secondary schools in England**. We derive this population from GIAS, which is the DfE's canonical register, then treat it as the *spine* every other dataset must join onto. Using one authoritative spine avoids double-counting schools that appear differently across files (a common problem when academies convert and receive a new URN).

In [ ]:
g = gias.copy()

phase_col  = find_col(g, "phaseofeducation_name", contains="phase")
status_col = find_col(g, "establishmentstatus_name", contains="status")
type_col   = find_col(g, "typeofestablishment_name", contains="typeofestablishment")

print(g[phase_col].value_counts(dropna=False).head(10), "\n")
print(g[status_col].value_counts(dropna=False).head(10))


In [ ]:
SECONDARY_PHASES = ["Secondary", "All-through", "Middle deemed secondary"]
INDEPENDENT_TYPES = g[type_col].dropna().unique()
INDEPENDENT_TYPES = [t for t in INDEPENDENT_TYPES if "independent" in t.lower()]

spine = g[
    g[status_col].str.contains("Open", case=False, na=False)
    & g[phase_col].isin(SECONDARY_PHASES)
    & ~g[type_col].isin(INDEPENDENT_TYPES)
].copy()

spine["urn"] = spine[URN_GIAS].str.strip()
spine = spine.drop_duplicates(subset="urn")
print(f"Spine: {len(spine):,} open state-funded secondary schools")


## 4. Data Merging / Homogenisation Strategy (Homogenisation step 2)

**Strategy (documented for the report):**

1. **Spine** = GIAS open state-funded secondary schools (unique `URN`).
2. **URN joins** (left joins onto the spine): DfE KS4 tables (per year) and Ofsted MI both key on `URN`.
3. **Geographic join**: GIAS supplies each school's **LSOA code**; IMD 2019 is keyed on LSOA, so deprivation attaches via `URN → LSOA → IMD`.
4. **Temporal homogenisation**: each academic year of KS4 data becomes a suffixed column set (`p8_2023`, `p8_2024`), producing one row per school (wide format) suitable for the ANN.
5. **Join-loss audit**: after every merge we record how many spine schools failed to match and why — this feeds directly into the data quality statement.

```
GIAS spine (URN, LSOA) ──URN──▶ KS4 2022/23
        │                └URN──▶ KS4 2023/24
        │                └URN──▶ Ofsted MI (latest inspection)
        └────LSOA──▶ IMD 2019 (deciles & domain scores)
```


In [ ]:
audit = []  # join-loss audit log

def left_join_audit(base, other, on_left, on_right, name, cols=None):
    other = other.copy()
    other[on_right] = other[on_right].astype(str).str.strip()
    if cols:
        other = other[[on_right] + cols]
    other = other.drop_duplicates(subset=on_right)
    merged = base.merge(other, how="left",
                        left_on=on_left, right_on=on_right,
                        suffixes=("", f"_{name}"))
    key_probe = cols[0] if cols else on_right
    matched = merged[key_probe].notna().sum() if key_probe in merged else np.nan
    audit.append({"merge": name, "spine_rows": len(base),
                  "matched": matched,
                  "match_rate_%": round(100 * matched / len(base), 1)})
    return merged


In [ ]:
# --- pick the analysis variables from each source (adjust to release names) ---
P8_COL   = find_col(ks4_23, "p8mea", contains="p8mea")      # Progress 8
A8_COL   = find_col(ks4_23, "att8scr", contains="att8")     # Attainment 8
FSM_COL  = find_col(ks4_23, "ptfsm6cla1a", contains="fsm")  # % disadvantaged
OE_COL   = find_col(ofsted, "overall_effectiveness", contains="effectiveness")
IMD_DEC  = find_col(imd, contains="index_of_multiple_deprivation_imd_decile")
IMD_SCORE= find_col(imd, contains="index_of_multiple_deprivation_imd_score")
INC_SCORE= find_col(imd, contains="income_score")
EDU_SCORE= find_col(imd, contains="education_skills_and_training_score")

df = spine[["urn", LSOA_GIAS]].rename(columns={LSOA_GIAS: "lsoa"})

for yr, ks4 in [("2023", ks4_23), ("2024", ks4_24)]:
    tmp = ks4.rename(columns={P8_COL: f"p8_{yr}", A8_COL: f"a8_{yr}",
                              FSM_COL: f"fsm_pct_{yr}"})
    df = left_join_audit(df, tmp, "urn", URN_KS4, f"ks4_{yr}",
                         cols=[f"p8_{yr}", f"a8_{yr}", f"fsm_pct_{yr}"])

df = left_join_audit(df, ofsted.rename(columns={OE_COL: "ofsted_grade"}),
                     "urn", URN_OFSTED, "ofsted", cols=["ofsted_grade"])

df = left_join_audit(df, imd.rename(columns={IMD_DEC: "imd_decile",
                                             IMD_SCORE: "imd_score",
                                             INC_SCORE: "income_score",
                                             EDU_SCORE: "edu_skills_score"}),
                     "lsoa", LSOA_IMD, "imd",
                     cols=["imd_decile", "imd_score",
                           "income_score", "edu_skills_score"])

pd.DataFrame(audit)


> **Interpretation to carry into the report:** KS4 match rates below 100% are expected — schools with no published results (new schools, suppressed small cohorts, closures mid-year) will not match. IMD match rates should be ~100% because every English postcode belongs to an LSOA; failures indicate malformed LSOA codes in GIAS. The audit table above *is* the evidence of a controlled merging strategy.

In [ ]:
# --- type conversion after merging (gov CSVs use "SUPP"/"NE"/"NA" markers) ---
SUPPRESSION_MARKERS = ["SUPP", "NE", "NA", "NP", "LOWCOV", ""]

num_cols = [c for c in df.columns
            if c.startswith(("p8_", "a8_", "fsm_pct_", "imd_", "income_", "edu_"))]
for c in num_cols:
    df[c] = (df[c].replace(SUPPRESSION_MARKERS, np.nan)
                  .str.replace("%", "", regex=False)
                  .astype(float))

df["imd_decile"] = df["imd_decile"].astype("Int64")

# Ofsted grade → ordered categorical / numeric code
grade_order = ["Outstanding", "Good", "Requires improvement", "Inadequate"]
df["ofsted_grade"] = (df["ofsted_grade"].str.strip().str.capitalize()
                        .replace({"Requires improvement": "Requires improvement",
                                  "Serious weaknesses": "Inadequate",
                                  "Special measures": "Inadequate"}))
df["ofsted_num"] = df["ofsted_grade"].map(
    {g: i for i, g in enumerate(grade_order[::-1], start=1)})  # 4=Outstanding

df.head()


## 5. Statistical Profiles

Per-variable profile: count, missing %, central tendency, dispersion, skewness. Skewness informs Phase-2 preprocessing (log transform / robust scaling before the ANN).

In [ ]:
profile_cols = ["p8_2023", "p8_2024", "a8_2023", "a8_2024",
                "fsm_pct_2023", "fsm_pct_2024",
                "imd_score", "income_score", "edu_skills_score", "ofsted_num"]
profile_cols = [c for c in profile_cols if c in df.columns]

prof = df[profile_cols].describe().T
prof["missing_%"] = (df[profile_cols].isna().mean() * 100).round(1)
prof["skew"] = df[profile_cols].skew().round(2)
prof.round(2)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_cols = ["p8_2024", "a8_2024", "fsm_pct_2024",
             "imd_score", "income_score", "edu_skills_score"]
for ax, c in zip(axes.ravel(), plot_cols):
    if c in df.columns:
        df[c].dropna().hist(bins=40, ax=ax)
        ax.set_title(c)
plt.suptitle("Distributions of key variables")
plt.tight_layout()
plt.show()


In [ ]:
# Target variable: Ofsted grade distribution (class imbalance check)
counts = df["ofsted_grade"].value_counts()
print(counts, "\n")
print("Majority-class share:",
      f"{counts.max() / counts.sum():.1%}",
      "→ accuracy alone is a misleading metric; use macro-F1 / balanced accuracy.")
counts.reindex(grade_order).plot(kind="bar", figsize=(6, 4),
                                 title="Ofsted Overall Effectiveness distribution")
plt.tight_layout(); plt.show()


## 6. Exemplary Correlations

Pearson (linear) **and** Spearman (monotonic) — reporting both is deliberate: where Spearman ≫ Pearson, the relationship is monotonic but non-linear, which is the empirical justification for using an ANN rather than linear regression.

In [ ]:
corr_cols = [c for c in ["p8_2024", "a8_2024", "fsm_pct_2024",
                         "imd_score", "income_score",
                         "edu_skills_score", "ofsted_num"] if c in df.columns]

pearson  = df[corr_cols].corr(method="pearson")
spearman = df[corr_cols].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (m, title) in zip(axes, [(pearson, "Pearson"), (spearman, "Spearman")]):
    im = ax.imshow(m, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(corr_cols))); ax.set_xticklabels(corr_cols, rotation=45, ha="right")
    ax.set_yticks(range(len(corr_cols))); ax.set_yticklabels(corr_cols)
    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            ax.text(j, i, f"{m.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"{title} correlation")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()


In [ ]:
# Exemplary bivariate relationships for the report
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(df["fsm_pct_2024"], df["p8_2024"], s=6, alpha=0.3)
axes[0].set_xlabel("% disadvantaged pupils (FSM6)"); axes[0].set_ylabel("Progress 8 (2023/24)")
axes[0].set_title("Disadvantage vs Progress 8")

df.boxplot(column="p8_2024", by="imd_decile", ax=axes[1])
axes[1].set_xlabel("IMD decile (1 = most deprived)"); axes[1].set_ylabel("Progress 8")
axes[1].set_title("Progress 8 by neighbourhood deprivation")

df.boxplot(column="p8_2024", by="ofsted_grade", ax=axes[2])
axes[2].set_title("Progress 8 by Ofsted grade"); axes[2].set_xlabel("")
plt.suptitle(""); plt.tight_layout(); plt.show()


In [ ]:
# Year-on-year stability of Progress 8 — how much signal history carries
both = df[["p8_2023", "p8_2024"]].dropna()
r = both.corr().iloc[0, 1]
plt.figure(figsize=(5.5, 5))
plt.scatter(both["p8_2023"], both["p8_2024"], s=6, alpha=0.3)
plt.xlabel("Progress 8 (2022/23)"); plt.ylabel("Progress 8 (2023/24)")
plt.title(f"Year-on-year P8 stability (r = {r:.2f}, n = {len(both):,})")
plt.tight_layout(); plt.show()


In [ ]:
# Multicollinearity among IMD domain scores — matters for SHAP interpretation
imd_domains = [c for c in ["imd_score", "income_score", "edu_skills_score"] if c in df.columns]
df[imd_domains].corr().round(2)


## 7. Missingness Analysis

Is missingness random, or systematically related to deprivation (informative missingness)?

In [ ]:
missing_tbl = (df[profile_cols].isna().mean() * 100).round(1).sort_values(ascending=False)
print("Missing % per variable:\n", missing_tbl, "\n")

# Does missing P8 correlate with deprivation?
df["p8_missing"] = df["p8_2024"].isna()
by_dec = df.groupby("imd_decile")["p8_missing"].mean() * 100
by_dec.plot(kind="bar", figsize=(7, 3.5),
            title="% schools with missing Progress 8 by IMD decile")
plt.ylabel("% missing"); plt.tight_layout(); plt.show()


## 8. Data Quality Statements

Fill / adapt these after running on the real files — this section maps one-to-one onto the assessors' third bullet point.

In [ ]:
# DO NOT CHANGE THE VARIABLE NAMES — adapt the text after running on real data
dataset_overview_notes = """
The homogenised dataset covers N state-funded secondary schools in England
(GIAS spine). KS4 performance data matched X% of the spine; unmatched schools
are predominantly newly opened schools and those with suppressed results
(cohort < threshold), coded 'SUPP' in DfE releases. Ofsted grades matched Y%;
missing values arise from schools not yet inspected under the current
framework and from the September 2024 suspension of single-word overall
judgements, which truncates the target variable for the most recent period.
IMD attachment via GIAS LSOA codes achieved ~Z% coverage. Progress 8 is
approximately normally distributed; FSM percentage and IMD scores are
right-skewed and will be transformed before ANN training. Missingness in
Progress 8 is [not] uniform across IMD deciles, indicating [no] informative
missingness.
"""

data_quality_statements = {
    "completeness": "See join-loss audit (Section 4) and missingness table (Section 7).",
    "consistency": ("Column names harmonised across releases; suppression markers "
                    "(SUPP/NE/NP) standardised to NaN; Ofsted sub-judgements mapped "
                    "to a single ordered scale."),
    "accuracy": ("All sources are official DfE/ONS statistics; URN and LSOA are "
                 "administered identifiers, minimising linkage error."),
    "timeliness": ("KS4 2022/23 and 2023/24 final releases; Ofsted MI latest monthly "
                   "extract; IMD 2019 is the most recent deprivation index."),
    "known_limitations": ("Academy converters receive new URNs, breaking historical "
                          "linkage; small-school P8 scores have wide confidence "
                          "intervals; IMD reflects school location, not necessarily "
                          "pupils' home neighbourhoods."),
}


## 9. Preliminary Insights, Limitations and Next Steps

In [ ]:
# DO NOT CHANGE THE VARIABLE NAMES — refine wording after running on real data
preliminary_insights = [
    "Progress 8 correlates negatively with FSM disadvantage, consistent with the literature reviewed in Section 2 of the report.",
    "Schools in the most deprived IMD deciles show both lower median P8 and greater variance, suggesting deprivation constrains but does not determine outcomes.",
    "Ofsted grades are heavily imbalanced toward 'Good', so classification metrics must be class-weighted.",
    "IMD domain scores are strongly intercorrelated, requiring care when interpreting SHAP attributions across correlated features.",
    "Year-on-year P8 stability indicates historical performance will be a strong (potentially dominant) predictor and should be ablated to isolate socio-economic effects.",
]

limitations = [
    "IMD measures the deprivation of the school's LSOA, not the catchment or pupil residence distribution.",
    "The 2024 suspension of single-word Ofsted judgements reduces target coverage for the latest year.",
]

next_steps = [
    "Extend the merge to absence, funding (per-pupil), pupil-teacher ratio and SEND datasets (Table 2 of the report).",
    "Build the sklearn preprocessing pipeline (imputation, scaling, encoding) mirroring the Week 2 practical, then proceed to ANN training with baselines.",
]
